
# Análisis de Exploración Petrolera - OilyGiant

## Contexto del Proyecto
La empresa OilyGiant desea desarrollar **200 nuevos pozos de petróleo** en la región con mayor potencial económico. Se busca utilizar modelos predictivos para estimar la producción potencial de nuevos pozos, seleccionar los más prometedores y analizar el riesgo asociado a cada decisión.

## Objetivos del Proyecto
1. Entrenar un modelo de regresión lineal para predecir el volumen de reservas.
2. Seleccionar los mejores pozos con base en las predicciones.
3. Evaluar qué región ofrece mayor beneficio total.
4. Usar bootstrapping para estimar riesgos y beneficios.

Los datos provienen de tres regiones distintas, y para cada pozo se conocen características técnicas (`f0`, `f1`, `f2`) y el volumen real de reservas (`product`).


In [1]:
import pandas as pd

# Cargar datos
df_0 = pd.read_csv("geo_data_0.csv")
df_1 = pd.read_csv("geo_data_1.csv")
df_2 = pd.read_csv("geo_data_2.csv")

# Vista rápida
df_0.head()


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647



### Descripción de columnas
- `id`: identificador único del pozo.
- `f0`, `f1`, `f2`: características técnicas del pozo (no se especifica el significado).
- `product`: volumen de reservas del pozo (en miles de barriles).


In [2]:
df_0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


In [4]:
X = df_0[['f0', 'f1', 'f2']]
X.head()

,f0,f1,f2
0,0.705745,-0.497823,1.221170
1,1.334711,-0.340164,4.365080
2,1.022732,0.151990,1.419926
3,-0.032172,0.139033,2.978566
4,1.988431,0.155413,4.751769


In [7]:
y = df_0['product']
y

0        105.280062
1         73.037750
2         85.265647
3        168.620776
4        154.036647
            ...    
99995    110.744026
99996    122.346843
99997     64.375443
99998     74.040764
99999    149.633246
Name: product, Length: 100000, dtype: float64

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

def entrenar_modelo(df):
    X = df.drop(columns=['id', 'product'])
    y = df['product']
    
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)
    
    model = LinearRegression()
    
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    
    return model, predictions, y_valid.reset_index(drop=True), rmse


In [9]:

model_0, pred_0, valid_0, rmse_0 = entrenar_modelo(df_0)
model_1, pred_1, valid_1, rmse_1 = entrenar_modelo(df_1)
model_2, pred_2, valid_2, rmse_2 = entrenar_modelo(df_2)

print("RMSE Región 0:", rmse_0)
print("RMSE Región 1:", rmse_1)
print("RMSE Región 2:", rmse_2)


RMSE Región 0: 37.75660035026169
RMSE Región 1: 0.8902801001028838
RMSE Región 2: 40.145872311342174



### Evaluación del modelo
El **RMSE** mide la diferencia promedio entre los valores reales y los predichos. Un valor menor implica una mejor predicción.

También se examina el volumen medio predicho, lo cual indica qué tanto se espera producir en promedio.


In [11]:

print("Media predicha Región 0:", pred_0.mean())
print("Media predicha Región 1:", pred_1.mean())
print("Media predicha Región 2:", pred_2.mean())


Media predicha Región 0: 92.39879990657768
Media predicha Región 1: 68.71287803913762
Media predicha Región 2: 94.77102387765939


In [12]:
pred_0

array([101.90101715,  78.21777385, 115.26690103, ...,  82.54439653,
        81.82668931,  93.12106221])

In [13]:
BARRIL_VALOR = 4500
POZOS = 200
INVERSION = 100000000
            

def calcular_ganancia(y_pred, y_real):
    indices_top = y_pred.sort_values(ascending=False).index[:POZOS]
    ganancia = y_real[indices_top].sum() * BARRIL_VALOR - INVERSION
    return ganancia

In [16]:
gan_0 = calcular_ganancia(pd.Series(pred_0), valid_0)
gan_1 = calcular_ganancia(pd.Series(pred_1), valid_1)
gan_2 = calcular_ganancia(pd.Series(pred_2), valid_2)

In [17]:
print("Ganancia Región 0:", gan_0)
print("Ganancia Región 1:", gan_1)
print("Ganancia Región 2:", gan_2)

Ganancia Región 0: 33591411.14462179
Ganancia Región 1: 24150866.966815114
Ganancia Región 2: 25985717.59374112


In [18]:
np.random.seed(42)

def bootstrap_ganancia(pred, real):
    ganancias = []
    for _ in range(1000):
        sample = pd.Series(np.random.choice(pred, size=500, replace=True))
        real_sample = pd.Series(np.random.choice(real, size=500, replace=True))
        top_indices = sample.sort_values(ascending=False).index[:POZOS]
        ganancias.append(real_sample[top_indices].sum() * BARRIL_VALOR - INVERSION)
    return ganancias



def analizar_riesgo(ganancias):
    ganancias = pd.Series(ganancias)
    media = ganancias.mean()
    intervalo = ganancias.quantile([0.025, 0.975])
    riesgo = (ganancias < 0).mean() * 100
    return media, intervalo, riesgo

gan_boot_0 = bootstrap_ganancia(pred_0, valid_0)
gan_boot_1 = bootstrap_ganancia(pred_1, valid_1)
gan_boot_2 = bootstrap_ganancia(pred_2, valid_2)

for i, g in enumerate([gan_boot_0, gan_boot_1, gan_boot_2]):
    media, intervalo, riesgo = analizar_riesgo(g)
    print(f"Región {i} -> Ganancia Prom: {media:.2f}, IC 95%: {intervalo.values}, Riesgo pérdida: {riesgo:.2f}%")


Región 0 -> Ganancia Prom: -17032724.60, IC 95%: [-22532131.77760003 -11815080.67529193], Riesgo pérdida: 100.00%
Región 1 -> Ganancia Prom: -38120428.82, IC 95%: [-43511526.57964028 -32673298.95305358], Riesgo pérdida: 100.00%
Región 2 -> Ganancia Prom: -14536834.96, IC 95%: [-19770087.21728544  -8752812.770656  ], Riesgo pérdida: 100.00%



## Conclusiones

- Se entrenó un modelo de regresión lineal por región para predecir el volumen de petróleo.
- Se comparó la ganancia esperada seleccionando los 200 pozos más prometedores.
- Se utilizó bootstrapping para simular riesgos y encontrar el **intervalo de confianza** y el **riesgo de pérdida**.
- La región óptima será aquella que **maximice la ganancia esperada** con un **riesgo menor al 2.5%**.

> Asegúrate de comunicar al equipo ejecutivo cuál es la región seleccionada y justificar la elección con base en las métricas mostradas.
